## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1065cb620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x106a70440>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'pro

In [5]:
model.invoke("Provide details about the moview avengers")

AIMessage(content='**The Avengers (2012)**  \n*(also known as *Marvel Studios: The Avengers* in some markets)*  \n\n| Category | Details |\n|----------|---------|\n| **Release date** | May 4, 2012 (U.S.) |\n| **Director** | Joss Whedon |\n| **Screenplay** | Joss Whedon (based on the 1963–1969 Marvel comic series) |\n| **Production companies** | Marvel Studios, Paramount Pictures |\n| **Distributor** | Paramount Pictures |\n| **Running time** | 143 minutes |\n| **Genre** | Action, Adventure, Science‑fiction, Super‑hero |\n| **Budget** | ~$220\u202fmillion |\n| **Box‑office gross** | $1.5\u202fbillion worldwide (as of 2024) |\n| **Language** | English (original) |\n| **Country** | United States |\n\n---\n\n## 1. Premise / Plot\n\nThe film brings together five of Marvel’s most iconic heroes—**Iron\u202fMan (Tony\u202fStark)**, **Captain America (Steve\u202fRogers)**, **Thor**, **Hulk (Bruce\u202fBanner)**, and **Black\u202fWidow (Natasha\u202fRomanoff)**—to form a team called the Avengers

In [6]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details.""" #docstring, provides metadata about model
    title: str = Field(..., description="The title of the movie") #the ...(elipses) mean this field is required 
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call function to get movie details. Use functions.Movie with title "Inception". Probably also need director, rating, year. We can provide those. Let\'s call function.', 'tool_calls': [{'id': 'fc_e6c08151-407c-432e-b7cd-13dc6912ce3b', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 164, 'total_tokens': 241, 'completion_time': 0.085967897, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.061976288, 'prompt_tokens_details': None, 'queue_time': 0.281109987, 'total_time': 0.147944185}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4a35f7bd1b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019faed9-c4ea-79e2-bd3c-3bc09c29da3f-0',

### Nested

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

What it does:
TypedDict defines the expected structure (keys and value types) for a dictionary and helps the LLM generate output in that format.

What it doesn't do:
It does not perform runtime validation or type checking—if the LLM returns missing fields or incorrect types, TypedDict won't catch or fix them.

In [9]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [11]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2008}

## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The person's name")
    email: str = Field(description="The person's email address")
    phone: str = Field(description="The person's phone number")

# Create the agent
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

# Invoke the agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})
contact = result["structured_response"]


In [15]:
print(contact)
print(contact.name)
print(contact.email)
print(contact.phone)

name='John Doe' email='john@example.com' phone='(555) 123-4567'
John Doe
john@example.com
(555) 123-4567
